In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from captum.attr import LayerIntegratedGradients, visualization as viz
import os
import matplotlib.pyplot as plt
import numpy as np

MODEL_PATH = "./final_ghost_detector"
BASE_MODEL_ID = "distilbert-base-uncased"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model on {device.upper()}...")

# 1. Load Model & Tokenizer
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID, num_labels=2
)
# Load the LoRA weights
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.to(device)
model.eval()
model.zero_grad()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

/home/andaburger/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/andaburger/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading model on CPU...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1598.74it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# 2. Setup Captum
try:
    embeddings_layer = model.base_model.model.distilbert.embeddings
except AttributeError:
    embeddings_layer = model.base_model.distilbert.embeddings

# B. predict_fn for Captum
def predict_fn(input_ids, attention_mask=None):
    return model(input_ids=input_ids, attention_mask=attention_mask).logits

# 2. initialize layer integrated gradients
lig = LayerIntegratedGradients(predict_fn, embeddings_layer)

In [ ]:
# 3. plotting function

def plot_token_importance(tokens, attributions, label_name, pred_class):
    tokens_filtered = []
    attrs_filtered = []
    
    for t, a in zip(tokens, attributions):
        if t not in ["[CLS]", "[SEP]", "[PAD]"]:
            tokens_filtered.append(t)
            attrs_filtered.append(a)
    
    tokens_filtered = np.array(tokens_filtered)
    attrs_filtered = np.array(attrs_filtered)
    
    if len(tokens_filtered) == 0:
        return

    #sort by absolute value to find most important tokens
    top_k = min(15, len(tokens_filtered))
    indices = np.argsort(np.abs(attrs_filtered))[-top_k:]
    
    top_tokens = tokens_filtered[indices]
    top_attrs = attrs_filtered[indices]
    
    plt.figure(figsize=(10, 8))
    colors = ['green' if x > 0 else 'red' for x in top_attrs]
    plt.barh(range(len(top_tokens)), top_attrs, color=colors)
    plt.yticks(range(len(top_tokens)), top_tokens, fontsize=12)
    plt.xlabel("Attribution Score", fontsize=12)
    plt.title(f"Top {top_k} Features for '{label_name}' Class\n(Model Predicted: {pred_class})", fontsize=14)
    plt.axvline(0, color='black', linestyle='--', linewidth=0.5)
    
    filename = f"feature_importance_{label_name}.png"
    plt.tight_layout()
    plt.savefig(filename)
    print(f"Saved feature importance plot to '{filename}'")
    plt.close()


In [ ]:
# We need to ensure input_ids are passed positionally to match predict_fn signature
def explain_text(text, label_name="AI"):
    target_idx = 1 if label_name == "AI" else 0
    
    # tokenize
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    # compute attributions
    attributions, delta = lig.attribute(
        inputs=input_ids,
        target=target_idx,
        additional_forward_args=(attention_mask), # This gets passed to predict_fn
        return_convergence_delta=True
    )

    # avg scores
    attributions_sum = attributions.sum(dim=-1).squeeze(0)
    attributions_sum = attributions_sum / torch.norm(attributions_sum)
    
    # Predict for visualization
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=1)
    
    pred_class_idx = torch.argmax(probs).item()
    confidence_score = probs[0, pred_class_idx].item() 
    true_label_score = probs[0, target_idx].item()     # confidence of the TRUE label specified
    
    pred_class = "AI" if pred_class_idx == 1 else "Human"

    print(f"\nText: {text[:60]}...")
    print(f"True Label: {label_name} | Predicted: {pred_class} ({confidence_score:.1%} confidence)")
    if pred_class != label_name:
        print(f"  (Model assigned {true_label_score:.1%} probability to the True Label '{label_name}')")

    # Visualize
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    # Zero out special tokens to avoid distraction in visualization
    for i, token in enumerate(tokens):
        if token in [tokenizer.cls_token, tokenizer.sep_token]:
            attributions_sum[i] = 0.0
            
    # Generate Bar Chart for top features
    plot_token_importance(
        tokens, 
        attributions_sum.detach().cpu().numpy(), 
        label_name, 
        pred_class
    )
    
    vis_data = viz.VisualizationDataRecord(
        word_attributions=attributions_sum.detach().cpu().numpy(),
        pred_prob=confidence_score,
        pred_class=pred_class,
        true_class=label_name,
        attr_class=label_name,
        attr_score=attributions_sum.sum(),
        raw_input_ids=tokens,
        convergence_score=delta
    )
    
    return vis_data


In [7]:
# --- RUNNING THE EXPERIMENT ---

# Sample 1: An "Imposter" paragraph
ai_text = """
Hushed it grew... utterly so, as if the very world did hold its breath in anticipation. Then commenced the snow, descending in delicate flakes, spectral in truth, drifting with such softness as to render all indistinct and white, a muffling embrace. Even the sharpest sounds did but... dwindle, being swallowed whole. And the wind, which had roared anon, became a mere whisper, scarcely audible, rustling the naked boughs. 'Twas not an absence of sound, but rather this... profound stillness that settled upon all, as a boon, a very blessing it seemed."""
# Sample 2: A Human paragraph 
human_text = """
I am very interested in problems involving mathematics, machine learning, NLP and quantita-
tive finance, probability and statistics, reinforcement learning and stochastic processes. I am

interested in doing research in these fields with like minded people, which I feel I can find in
Precog. I feel I would be a strong fit because of my interdisciplinary orientation. I actively seek
feedback to refine my understanding. I feel like this is a place where my interests and work
ethic naturally belong."""

print("\n--- Generating Saliency Maps ---")
vis_ai = explain_text(ai_text, label_name="AI")
vis_human = explain_text(human_text, label_name="Human")

# SAVE TO HTML
print("\nSaving visualization to 'ghost_explanation.html'...")
html_obj = viz.visualize_text([vis_ai, vis_human])

with open("ghost_explanation.html", "w", encoding="utf-8") as f:
    f.write(html_obj.data)

print("Done! Open 'ghost_explanation.html' in your browser.")

"""
some notable things are picking up of certain words like "Family" and even for some r, also exclamation marks and punctuations are being flagged as human in this because of 
books having more of these. A word like 'london' or some other location or to be heavily used in the human text and not in AI text(only used once in all the 500 paragraphs) """



--- Generating Saliency Maps ---

Text: 
Hushed it grew... utterly so, as if the very world did hold...
True Label: AI | Predicted: Human (95.4% confidence)
  (Model assigned 4.6% probability to the True Label 'AI')
Saved feature importance plot to 'feature_importance_AI.png'

Text: 
I am very interested in problems involving mathematics, mac...
True Label: Human | Predicted: AI (81.4% confidence)
  (Model assigned 18.6% probability to the True Label 'Human')
Saved feature importance plot to 'feature_importance_Human.png'

Saving visualization to 'ghost_explanation.html'...


Done! Open 'ghost_explanation.html' in your browser.


'\nsome notable things are picking up of certain words like "Family" and even for some r, also exclamation marks and punctuations are being flagged as human in this because of \nbooks having more of these. A word like \'london\' or some other location or to be heavily used in the human text and not in AI text(only used once in all the 500 paragraphs) '